# Trainable Corner-Regressor (CNN kecil, prediksi 4 titik langsung)

Ini alternatif buat gantiin (atau jadi fallback dari) pipeline classical CV (Canny + contour + scoring)
di notebook sebelumnya. Alih-alih cari kontur + skoring heuristik, di sini kita **train CNN kecil yang
langsung meregresi 8 angka** (x,y untuk 4 corner: top-left, top-right, bottom-right, bottom-left),
dinormalisasi ke [0,1] relatif ke ukuran gambar.

**Kenapa ini bisa lebih baik dari classical CV:** classical pipeline gampang ketipu sama tepi lain yang
kontras/tajam (poster di background, headshot inset di dalam kartu) karena dia nggak punya pemahaman
semantik. CNN regressor belajar langsung dari label "ini beneran corner kartu", jadi bisa lebih robust
di kasus cluttered/low-contrast — asal datanya cukup dan representatif.

**Kenapa ini juga punya risiko:** CNN regressor cuma sekuat label & data trainingnya. Kalau dataset kamu
kecil atau kurang variatif (angle, lighting, background), dia bisa overfit / gagal generalize ke kasus
yang belum pernah diliat.

**Struktur notebook:**
1. **Auto-labeling** — kamu belum punya ground-truth 4 corner, jadi label di-generate otomatis pakai
   pipeline classical CV (`detect_document`) dari notebook sebelumnya, disimpan ke `corners_labels.json`.
   Cuma kandidat dengan skor tinggi yang dipakai jadi training label.
2. **Dataset & augmentasi** — load gambar + label, resize ke ukuran tetap buat network, augmentasi
   (rotasi, brightness/contrast, sedikit translasi) — augmentasi geometris ikut transform label juga.
3. **Model** — CNN kecil (beberapa conv block + GAP + FC head), output 8 nilai (4 titik × x,y).
4. **Training loop** — MSE/SmoothL1 loss, train/val split, checkpoint model terbaik.
5. **Inference & evaluasi visual** — load model, prediksi corner di gambar baru, warp pakai
   `four_point_transform` yang sama kayak notebook classical CV, plot before/after.
6. **(Opsional) Ensemble/fallback** — gabungin dengan classical CV: kalau confidence classical rendah,
   pakai CNN, atau sebaliknya — ide dan kerangka kodenya di paling bawah, tinggal kamu isi threshold-nya.

Catatan: aku nggak run notebook ini (sesuai permintaan) — semua path/parameter di bawah perlu kamu
sesuaikan (`IMAGES_DIR`, `LABELS_PATH`, jumlah epoch, dst) terus jalanin lokal.

In [8]:
from pathlib import Path
import json
import time
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

plt.rcParams['figure.facecolor'] = 'white'
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## 1. Konfigurasi path

Ganti `IMAGES_DIR` ke folder foto dokumen/ID card kamu (bisa dataset KYC yang sama kayak notebook
classical CV). `LABELS_PATH` adalah file JSON tempat label 4-corner disimpan — kalau belum ada, akan
dibuat otomatis pas kamu mulai labeling di Section 2.

Format label per gambar: `{"nama_file.jpg": [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]}` dalam urutan
**top-left, top-right, bottom-right, bottom-left**, dalam koordinat pixel di gambar ASLI (bukan resized).

In [9]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'preprocessed' / 'baseline' / 'normalized').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
IMAGES_DIR = PROJECT_ROOT / 'preprocessed' / 'baseline' / 'normalized'  # <-- sesuaikan kalau strukturmu beda
LABELS_PATH = PROJECT_ROOT / "corners_labels.json"  # <-- file label, auto-dibuat kalau belum ada

IMG_SIZE = 256          # ukuran input network (persegi, gambar di-resize+letterbox ke ini)
VAL_FRACTION = 0.15

image_paths = sorted([p for p in Path(IMAGES_DIR).glob("*")
                       if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
print(f"Ketemu {len(image_paths)} gambar di {IMAGES_DIR}")

if Path(LABELS_PATH).exists():
    with open(LABELS_PATH) as f:
        labels = json.load(f)
else:
    labels = {}
print(f"Sudah ada label buat {len(labels)} gambar")

Ketemu 732 gambar di e:\comp\compfest_final_proj\preprocessed\baseline\normalized
Sudah ada label buat 0 gambar


## 2. Auto-labeling pakai classical CV pipeline

Karena belum ada ground-truth 4-corner, di sini kita **generate label otomatis** pakai pipeline
classical CV (`detect_document`) dari notebook sebelumnya — Canny + adaptive threshold + multi-scale
contour scoring. Ini dipakai sebagai **pseudo-label**: kandidat yang skornya di atas `AUTO_LABEL_MIN_SCORE`
dianggap cukup dipercaya buat jadi training target CNN.

Konsekuensi penting yang perlu kamu sadari: **CNN yang dilatih dari pseudo-label ini nggak akan lebih
akurat dari classical CV yang jadi sumber labelnya** — paling bagus dia belajar niru classical CV,
plus dapet bonus lebih robust di kasus interpolasi (angle/lighting di antara training samples) karena
CNN memang cenderung lebih smooth/general dibanding heuristik contour yang bisa patah di kasus edge-case.
Kalau butuh CNN yang beneran lebih akurat dari classical CV, ujung-ujungnya tetep butuh sebagian label
manual (curated) — bukan cuma auto dari pipeline yang sama.

Gambar yang skornya di bawah threshold (classical CV sendiri nggak yakin) **di-skip**, nggak dipaksa
masuk training set — biar CNN nggak ikut belajar dari label yang salah.

In [10]:
def order_points(pts):
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]      # top-left
    rect[2] = pts[np.argmax(s)]      # bottom-right
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]   # top-right
    rect[3] = pts[np.argmax(diff)]   # bottom-left
    return rect


def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect
    widthA, widthB = np.linalg.norm(br - bl), np.linalg.norm(tr - tl)
    maxW = max(int(widthA), int(widthB))
    heightA, heightB = np.linalg.norm(tr - br), np.linalg.norm(tl - bl)
    maxH = max(int(heightA), int(heightB))
    dst = np.array([[0, 0], [maxW - 1, 0], [maxW - 1, maxH - 1], [0, maxH - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(image, M, (maxW, maxH))


def corner_angle_score(rect):
    pts = [rect[0], rect[1], rect[2], rect[3]]
    angles = []
    for i in range(4):
        p_prev, p_cur, p_next = pts[i - 1], pts[i], pts[(i + 1) % 4]
        v1, v2 = p_prev - p_cur, p_next - p_cur
        n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if n1 < 1e-3 or n2 < 1e-3:
            return 0.0
        cos_a = np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)
        angles.append(np.degrees(np.arccos(cos_a)))
    deviation = np.mean([abs(a - 90) for a in angles])
    return max(0.0, 1 - deviation / 35)


def score_candidate(corners, frame_area, expected_ratio=1.58, ratio_tolerance=0.6):
    rect = order_points(corners.reshape(4, 2).astype(np.float32))
    w = np.linalg.norm(rect[1] - rect[0])
    h = np.linalg.norm(rect[3] - rect[0])
    if min(w, h) < 1:
        return 0.0, 0.0, 0.0
    area_frac = (w * h) / frame_area
    ratio = max(w, h) / min(w, h)
    ratio_score = max(0.0, 1 - abs(ratio - expected_ratio) / ratio_tolerance)
    cnt_area = cv2.contourArea(corners.astype(np.float32))
    rectangularity = min(1.0, cnt_area / max(w * h, 1))
    angle_score = corner_angle_score(rect)
    size_score = min(1.0, area_frac / 0.30)
    score = 0.45 * size_score + 0.20 * rectangularity + 0.15 * ratio_score + 0.20 * angle_score
    return score, ratio, area_frac


def detect_document(img, min_area_frac=0.10, border_density_thresh=0.05, min_score=0.45,
                     scales=(700, 1100, 1500)):
    '''Sama persis dengan pipeline classical CV di notebook sebelumnya — dipakai di sini
    khusus buat auto-generate pseudo-label corner, bukan buat inference final.'''
    h, w = img.shape[:2]
    candidates = []
    for target in scales:
        scale = target / max(h, w)
        if scale >= 1.0 and target != scales[0]:
            continue
        small = cv2.resize(img, (max(1, int(w * scale)), max(1, int(h * scale))))
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (7, 7), 0)
        sh, sw = small.shape[:2]
        frame_area = sh * sw

        edge_maps = []
        for lo, hi in [(20, 80), (30, 100), (50, 150), (75, 200), (100, 250)]:
            edged = cv2.Canny(blur, lo, hi)
            edged = cv2.dilate(edged, None, iterations=3)
            edged = cv2.erode(edged, None, iterations=2)
            edge_maps.append(edged)
        adapt = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 25, 5)
        adapt = cv2.dilate(adapt, None, iterations=2)
        edge_maps.append(adapt)

        for edged in edge_maps:
            cnts, _ = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for c in cnts:
                area = cv2.contourArea(c)
                if area < min_area_frac * frame_area:
                    continue
                hull = cv2.convexHull(c)
                peri = cv2.arcLength(hull, True)
                approx = None
                for eps_frac in [0.02, 0.03, 0.05, 0.08]:
                    a = cv2.approxPolyDP(hull, eps_frac * peri, True)
                    if len(a) == 4:
                        approx = a
                        break
                if approx is None:
                    rect = cv2.minAreaRect(hull)
                    approx = cv2.boxPoints(rect).astype(np.int32).reshape(-1, 1, 2)
                score, ratio, area_frac = score_candidate(approx.reshape(4, 2), frame_area)
                candidates.append((score, ratio, area_frac, approx, scale))

    best = max(candidates, key=lambda x: x[0]) if candidates else None
    if best is None or best[0] < min_score:
        return {"success": False, "corners": None, "score": 0.0}

    score, ratio, area_frac, approx, scale = best
    corners_full = (approx.reshape(4, 2) / scale).astype(np.float32)
    rect = order_points(corners_full)  # pastiin urutan TL,TR,BR,BL konsisten
    return {"success": True, "corners": rect, "score": score}


AUTO_LABEL_MIN_SCORE = 0.55  # naikkan kalau mau lebih strict/bersih, turunkan kalau mau lebih banyak data

new_labels = 0
skipped_low_score = 0
for p in image_paths:
    if p.name in labels:
        continue
    img = cv2.imread(str(p))
    if img is None:
        continue
    result = detect_document(img, min_score=AUTO_LABEL_MIN_SCORE)
    if result["success"]:
        labels[p.name] = result["corners"].tolist()
        new_labels += 1
    else:
        skipped_low_score += 1

with open(LABELS_PATH, "w") as f:
    json.dump(labels, f)

print(f"Auto-label baru: {new_labels} | di-skip (skor < {AUTO_LABEL_MIN_SCORE}): {skipped_low_score}")
print(f"Total label sekarang: {len(labels)}")

Auto-label baru: 620 | di-skip (skor < 0.55): 112
Total label sekarang: 620


## 3. Dataset & augmentasi

`CornerDataset` load gambar, resize+letterbox ke `IMG_SIZE x IMG_SIZE` (aspect ratio dijaga, sisanya
padding, biar corner nggak distorsi), lalu transform koordinat label ikut resize+padding itu.

Augmentasi yang dipakai (geometris, jadi label ikut ditransform juga):
- Rotasi kecil random (±15°)
- Translasi kecil random
- Horizontal flip (dengan re-order titik biar tetep TL/TR/BR/BL yang benar)

Plus augmentasi non-geometris (nggak ubah label): brightness/contrast jitter, sedikit gaussian noise/blur
— biar model nggak terlalu bergantung sama pencahayaan/ketajaman spesifik.

In [ ]:
def letterbox(img, size):
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (nw, nh))
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    top = (size - nh) // 2
    left = (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas, scale, left, top


def transform_points(pts, scale, left, top):
    pts = np.array(pts, dtype=np.float32)
    pts = pts * scale
    pts[:, 0] += left
    pts[:, 1] += top
    return pts


class CornerDataset(Dataset):
    '''pts disimpan/dikembalikan urutan TL, TR, BR, BL, dinormalisasi ke [0,1] relatif IMG_SIZE.'''

    def __init__(self, samples, images_dir, img_size=256, augment=False):
        self.samples = samples  # list of (filename, pts_pixel_original)
        self.images_dir = Path(images_dir)
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def _augment(self, img, pts):
        h, w = img.shape[:2]
        # rotasi + translasi kecil di sekitar center
        angle = random.uniform(-15, 15)
        tx, ty = random.uniform(-0.05, 0.05) * w, random.uniform(-0.05, 0.05) * h
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        M[0, 2] += tx
        M[1, 2] += ty
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT101)
        ones = np.ones((pts.shape[0], 1))
        pts_h = np.hstack([pts, ones])
        pts = (M @ pts_h.T).T

        if random.random() < 0.5:
            img = cv2.flip(img, 1)
            pts[:, 0] = w - pts[:, 0]
            # flip horizontal menukar TL<->TR dan BL<->BR biar urutan tetep konsisten
            pts = pts[[1, 0, 3, 2]]

        if random.random() < 0.7:
            alpha = random.uniform(0.8, 1.2)   # contrast
            beta = random.uniform(-20, 20)     # brightness
            img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        if random.random() < 0.3:
            k = random.choice([3, 5])
            img = cv2.GaussianBlur(img, (k, k), 0)

        return img, pts

    def __getitem__(self, idx):
        fname, pts_orig = self.samples[idx]
        img = cv2.imread(str(self.images_dir / fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pts = np.array(pts_orig, dtype=np.float32)

        if self.augment:
            img, pts = self._augment(img, pts)

        canvas, scale, left, top = letterbox(img, self.img_size)
        pts = transform_points(pts, scale, left, top)
        pts = np.clip(pts, 0, self.img_size - 1)

        pts_norm = pts / self.img_size  # [0,1]
        img_t = torch.from_numpy(canvas).permute(2, 0, 1).float() / 255.0
        target_t = torch.from_numpy(pts_norm.reshape(-1).astype(np.float32))  # shape (8,)
        return img_t, target_t


# --- build sample list & split ---
all_samples = [(fname, pts) for fname, pts in labels.items()
               if (Path(IMAGES_DIR) / fname).exists()]
random.shuffle(all_samples)

if len(all_samples) < 5:
    raise RuntimeError(
        f"Cuma ada {len(all_samples)} sample berlabel valid (labels dict: {len(labels)} entri, "
        f"LABELS_PATH: {LABELS_PATH}). Jalanin dulu label_images(...) di Section 2 sampai punya "
        f"cukup gambar berlabel sebelum lanjut ke sini."
    )

n_val = max(1, int(len(all_samples) * VAL_FRACTION))
val_samples = all_samples[:n_val]
train_samples = all_samples[n_val:]
print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

train_ds = CornerDataset(train_samples, IMAGES_DIR, IMG_SIZE, augment=True)
val_ds = CornerDataset(val_samples, IMAGES_DIR, IMG_SIZE, augment=False)

BATCH_SIZE = min(16, max(1, len(train_samples)))  # biar nggak error kalau data masih sedikit
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Train: 527, Val: 93


## 4. Model — CNN kecil

Backbone konvolusi ringan (5 blok conv+BN+ReLU+maxpool, channel naik bertahap 32→256), diakhiri
Global Average Pooling + 2 FC layer, output 8 nilai (sigmoid biar otomatis di range [0,1], match sama
`pts_norm`). Ini didesain ringan supaya bisa dilatih di CPU/GPU kecil dalam waktu wajar — kalau
akurasi kurang dan kamu punya compute lebih, gampang di-swap ke backbone pretrained (mis. MobileNetV3
dari `torchvision.models`) tinggal ganti bagian `self.backbone`.

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.net(x)


class CornerRegressor(nn.Module):
    '''Input: (B, 3, IMG_SIZE, IMG_SIZE). Output: (B, 8) = [tl_x,tl_y,tr_x,tr_y,br_x,br_y,bl_x,bl_y],
    semua dinormalisasi [0,1] via sigmoid.'''

    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            ConvBlock(3, 32),     # 256 -> 128
            ConvBlock(32, 64),    # 128 -> 64
            ConvBlock(64, 128),   # 64 -> 32
            ConvBlock(128, 256),  # 32 -> 16
            ConvBlock(256, 256),  # 16 -> 8
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 8),
        )

    def forward(self, x):
        feat = self.backbone(x)
        out = self.head(feat)
        return torch.sigmoid(out)


model = CornerRegressor().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_params:,}")

Total params: 2,389,288


## 5. Training loop

Loss pakai **Smooth L1** (Huber) daripada plain MSE — lebih tahan terhadap label noisy/klik yang
kurang presisi (yang lumayan mungkin kejadian pas manual labeling), tapi masih differentiable mulus
di sekitar 0 nggak kayak plain L1. Optimizer Adam + `ReduceLROnPlateau` berdasarkan val loss, plus
checkpoint model terbaik ke `best_corner_model.pt`.

Metrik tambahan yang di-track: **rata-rata pixel error per corner** (di skala `IMG_SIZE`) biar lebih
gampang diinterpretasi dibanding angka loss mentah.

In [13]:
def mean_corner_pixel_error(pred_norm, target_norm, img_size):
    pred = pred_norm.view(-1, 4, 2) * img_size
    target = target_norm.view(-1, 4, 2) * img_size
    dist = torch.norm(pred - target, dim=-1)  # (B, 4)
    return dist.mean().item()


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_err, n_batches = 0.0, 0.0, 0
    with torch.set_grad_enabled(is_train):
        for imgs, targets in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            preds = model(imgs)
            loss = criterion(preds, targets)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            total_err += mean_corner_pixel_error(preds.detach(), targets, IMG_SIZE)
            n_batches += 1
    return total_loss / max(n_batches, 1), total_err / max(n_batches, 1)


EPOCHS = 60
LR = 1e-3

criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "train_err": [], "val_err": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_err = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_err = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_err"].append(train_err)
    history["val_err"].append(val_err)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_corner_model.pt")
        tag = " (saved)"
    else:
        tag = ""

    print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.5f} val_loss={val_loss:.5f} "
          f"| train_err={train_err:5.1f}px val_err={val_err:5.1f}px{tag}")

RuntimeError: DataLoader worker (pid(s) 16308, 17288) exited unexpectedly

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss (Smooth L1)")
axes[0].legend()

axes[1].plot(history["train_err"], label="train")
axes[1].plot(history["val_err"], label="val")
axes[1].set_title(f"Mean corner pixel error (di skala {IMG_SIZE}px)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Inference & evaluasi visual

`predict_corners()` ambil path gambar, jalanin lewat model, terus balikin corner dalam koordinat
**pixel gambar asli** (undo letterbox). `four_point_transform` & `order_points` sama persis kayak yang
dipakai di notebook classical CV, jadi hasil kedua pipeline langsung bisa dibandingin apple-to-apple.

In [ ]:
def predict_corners(model, img_path, img_size=IMG_SIZE, device=DEVICE):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    canvas, scale, left, top = letterbox(img_rgb, img_size)
    img_t = torch.from_numpy(canvas).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    img_t = img_t.to(device)

    model.eval()
    with torch.no_grad():
        pred = model(img_t).cpu().numpy().reshape(4, 2) * img_size

    # undo letterbox -> koordinat pixel gambar asli
    pred[:, 0] = (pred[:, 0] - left) / scale
    pred[:, 1] = (pred[:, 1] - top) / scale
    return img_bgr, pred.astype(np.float32)


def show_prediction(model, img_path):
    img_bgr, corners = predict_corners(model, img_path)
    warped = four_point_transform(img_bgr, corners)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    disp = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    pts_int = corners.astype(int)
    for i in range(4):
        cv2.line(disp, tuple(pts_int[i]), tuple(pts_int[(i + 1) % 4]), (255, 0, 0), 3)
    for p in pts_int:
        cv2.circle(disp, tuple(p), 6, (0, 255, 0), -1)
    axes[0].imshow(disp)
    axes[0].set_title(f"Predicted corners — {Path(img_path).name}")
    axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Warped (perspective-corrected)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


# load best checkpoint sebelum eval
model.load_state_dict(torch.load("best_corner_model.pt", map_location=DEVICE))

# contoh: coba di beberapa gambar val
for fname, _ in val_samples[:5]:
    show_prediction(model, Path(IMAGES_DIR) / fname)

## 7. (Opsional) Kombinasi dengan classical CV pipeline

Kalau mau ensemble/fallback antara CNN dan pipeline classical CV (notebook sebelumnya): pola yang umum
dipakai adalah **pakai classical CV dulu** (cepat, nggak butuh training, dan `score_candidate()`-nya
udah kasih confidence eksplisit), lalu **fallback ke CNN** kalau confidence classical rendah — atau
sebaliknya kalau kamu udah observasi CNN lebih reliable di dataset kamu.

Beberapa cara nge-tune ini setelah kamu punya kedua model:
- Bandingin `mean_corner_pixel_error` (val) CNN vs distribusi `score` dari `detect_document()` di
  gambar yang sama, cek di kondisi apa masing-masing menang/kalah (background cluttered? low-light?).
- Kalau CNN robust ke clutter tapi classical CV lebih presisi di kasus bersih (biasanya kejadian,
  karena regression CNN kecil punya batas presisi sub-pixel), pertimbangkan pakai **CNN buat rough
  localization** lalu **classical edge-refinement lokal di sekitar tiap corner prediksi** (misal cari
  garis lurus terdekat dalam radius kecil) buat gabungin kekuatan dua-duanya — tapi ini langkah lanjutan,
  belum diimplementasikan di sini.

```python
# kerangka kasar, isi sendiri sesuai kebutuhan:
# from document_detection_5 import detect_document  # pipeline classical CV
#
# def detect_hybrid(img, cnn_model, classical_min_score=0.45):
#     classical = detect_document(img)
#     if classical["success"] and classical["score"] >= classical_min_score:
#         return classical
#     # fallback ke CNN
#     ...
```